# rounds

> Capture, review, accept, and start curated tool-use rounds.

In [ ]:
#| default_exp rounds

In [ ]:
#| export
from dataclasses import asdict
from pathlib import Path
import json, shlex, subprocess

from aidialog.dialog import Dialog, snote, sprompt
from aidialog.hist import chat2dlg, dlg2chat
from aidialog.ipynb import read_ipynb, write_ipynb
from aidialog.msg_parts import Msg, Text, ToolUse, ToolResult
from fastcore.script import call_parse

from drona.core import RAMABANA_HISTORY, assess_history, drona_version, read_history

## Capture a saved session

A capture reads completed Ramabana turns. It does not observe a live process. The output is an Aidialog notebook that Leela can open directly.

In [ ]:
#| export
REVIEW_KEY = 'drona'

def latest_session(turns):
    "The newest session id in `turns`."
    return turns[-1].get('session') if turns else None

def _turn_msgs(turn):
    "One Ramabana archive turn as typed Aidialog messages."
    user = Msg('user', [Text(turn.get('prompt', ''))])
    replies = []
    for i, action in enumerate(turn.get('activity') or []):
        call_id = action.get('action_id') or action.get('id') or f'call_{i}'
        tool = action.get('tool', '')
        args = action.get('args') or {}
        replies += [
            Msg('assistant', [ToolUse(id=call_id, name=tool, arguments=args)]),
            Msg('tool', [ToolResult(id=call_id, name=tool, arguments=args,
                                    text=str(action.get('detail', '')))]),
        ]
    if text := turn.get('reply', ''): replies.append(Msg('assistant', [Text(text)]))
    return [user, *replies]

def capture(
    output,                 # review notebook path
    session='latest',       # Ramabana session id, prefix, or `latest`
    history=RAMABANA_HISTORY, # Ramabana history path
    name=None,              # round name; output stem when omitted
):
    "Capture one persisted Ramabana session as a review notebook."
    turns = read_history(history)
    if not turns: raise ValueError('Ramabana history has no completed turns')
    if session == 'latest': session = latest_session(turns)
    matches = [t for t in turns if str(t.get('session', '')).startswith(session)]
    ids = {t.get('session') for t in matches}
    if not matches: raise ValueError(f'No Ramabana session matches {session!r}')
    if len(ids) > 1: raise ValueError(f'Ramabana session prefix {session!r} is ambiguous')
    assessment = assess_history(matches)
    msgs = [m for turn in matches for m in _turn_msgs(turn)]
    dlg = chat2dlg(msgs, name or Path(output).stem, mx=None)
    report = {'version': drona_version(), 'session': matches[0].get('session'),
              'status': 'review', 'score': assessment.score,
              'findings': [asdict(f) for f in assessment.findings]}
    dlg.meta[REVIEW_KEY] = report
    dlg.mk_message('# Drona review\n\nEdit this dialog in Leela. Remove poor routes and sensitive content. Set `reviewer` when accepting.',
                   idx=0, msg_type=snote, skipped=1, meta={REVIEW_KEY: report})
    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    write_ipynb(dlg, output)
    return output

In [ ]:
import tempfile
capture_dir = Path(tempfile.mkdtemp())
history = capture_dir/'history.jsonl'
turn = {
    'session': 'agent_example', 'prompt': 'Use FOSSICK to research a GitHub repository.',
    'reply': 'The repository says demonstrations improve tool use.',
    'activity': [{'id': 'a1', 'tool': 'web_search', 'args': {'query': 'repo'},
                  'detail': 'generic result', 'ok': True}],
}
history.write_text(json.dumps(turn) + '\n')
review = capture(capture_dir/'github.ipynb', history=history)
dlg = read_ipynb(review)
assert dlg.meta[REVIEW_KEY]['status'] == 'review'
assert dlg[0].skipped and dlg[1].msg_type == sprompt
review

## Accept a reviewed round

Acceptance is explicit. Drona validates the dialog round trip and records the reviewer, source session, package version, and round revision in notebook metadata.

In [ ]:
#| export
def _review_prompts(dlg): return [m for m in dlg if m.msg_type == sprompt and not m.skipped]

def compiled_history(source):
    "Canonical Aidialog history from an accepted round."
    dlg = read_ipynb(source)
    if not dlg: raise ValueError(f'Could not read dialog {source}')
    meta = dlg.meta.get(REVIEW_KEY) or {}
    if meta.get('status') != 'accepted': raise ValueError('Round has not been accepted')
    clean = Dialog(_review_prompts(dlg), name=dlg.name)
    return dlg2chat(clean, plain=True)

def accept(
    source,         # reviewed dialog notebook
    reviewer,       # person accepting the round
    output=None,    # compiled JSON path; `<source>.json` when omitted
):
    "Accept a reviewed dialog and write its canonical history."
    if not reviewer.strip(): raise ValueError('reviewer is required')
    dlg = read_ipynb(source)
    if not dlg: raise ValueError(f'Could not read dialog {source}')
    prompts = _review_prompts(dlg)
    if not prompts: raise ValueError('Round contains no prompt turns')
    clean = Dialog(prompts, name=dlg.name)
    history = dlg2chat(clean, plain=True)
    if not history or history[0].role != 'user': raise ValueError('Round must start with a user turn')
    meta = dict(dlg.meta.get(REVIEW_KEY) or {})
    meta.update(status='accepted', reviewer=reviewer, accepted_version=drona_version())
    dlg.meta[REVIEW_KEY] = meta
    dlg.save(source)
    output = Path(output) if output else Path(source).with_suffix('.json')
    output.write_text(json.dumps({'meta': meta, 'history': _history_dicts(history)}, indent=2))
    return output

def _part_dict(part):
    data = {'type': part.type.value if hasattr(part.type, 'value') else str(part.type)}
    data.update({k:v for k,v in vars(part).items() if v not in (None, False, {}, []) and k != 'raw'})
    return data

def _history_dicts(history):
    return [{'role': m.role, 'content': [_part_dict(p) for p in m.content]} for m in history]

In [ ]:
compiled = accept(review, reviewer='Karthik')
accepted = json.loads(compiled.read_text())
assert accepted['meta']['status'] == 'accepted'
history2 = compiled_history(review)
assert history2[0].role == 'user'
assert any(isinstance(p, ToolUse) for m in history2 for p in m.content)
compiled

## Start a Ramabana session

Ramabana does not accept prepared history on its command line. `start_round` therefore sends the accepted round as a bootstrap prompt. Ramabana saves that turn, then Drona resumes the same session for normal work.

In [ ]:
#| export
def bootstrap_prompt(source):
    "A compact prompt containing one accepted worked round."
    history = compiled_history(source)
    rows = ['The following reviewed Drona round demonstrates the tool route to follow.']
    for m in history:
        if m.role == 'user': rows.append(f'User: {m.text}')
        elif m.role == 'assistant':
            for p in m.content:
                if isinstance(p, Text) and p.text: rows.append(f'Assistant: {p.text}')
                elif isinstance(p, ToolUse): rows.append(f'Assistant tool: {p.name}({json.dumps(p.arguments, sort_keys=True)})')
        elif m.role == 'tool':
            for p in m.content:
                if isinstance(p, ToolResult): rows.append(f'Tool result: {p.text}')
    rows.append('Reply with exactly: DRONA_READY')
    return '\n\n'.join(rows)

def start_commands(
    source,   # accepted review notebook
    root='.', # Ramabana root
    model=None, # optional model name
):
    "The bootstrap and resume commands for an accepted round."
    prompt = bootstrap_prompt(source)
    cmd = ['ramabana', '--root', root]
    if model: cmd += ['--model', model]
    cmd += [prompt]
    return cmd, ['ramabana', '--root', root, '--resume', 'latest']

def start_round(source, root='.', model=None, launch=False):
    "Prepare or launch Ramabana with an accepted Drona round."
    first, resume = start_commands(source, root, model)
    if not launch: return {'bootstrap': first, 'resume': resume}
    subprocess.run(first, check=True)
    return subprocess.run(resume, check=True).returncode

In [ ]:
commands = start_round(review, root='/tmp/project')
assert commands['bootstrap'][:3] == ['ramabana', '--root', '/tmp/project']
assert commands['resume'][-1] == 'latest'
assert commands['bootstrap'][-1].endswith('DRONA_READY')
commands

## Command line

The command line separates capture, acceptance, and launch. `start` prints commands unless `--launch` is set.

In [ ]:
#| export
def _print_commands(commands):
    for name, command in commands.items(): print(f'{name}: {shlex.join(command)}')

@call_parse
def capture_cli(output: str, session: str='latest', history: str=str(RAMABANA_HISTORY), name: str=None):
    "Capture a saved Ramabana session for review."
    print(capture(output, session, history, name))

@call_parse
def accept_cli(source: str, reviewer: str='', output: str=None):
    "Accept a reviewed round and compile its history."
    print(accept(source, reviewer, output))

@call_parse
def start_cli(source: str, root: str='.', model: str=None, launch: bool=False):
    "Prepare or launch Ramabana with an accepted round."
    result = start_round(source, root, model, launch)
    if isinstance(result, dict): _print_commands(result)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()